# HUFAC-Net: Hierarchical Unsupervised Fuzzy Attention Convolutional Network

Google Colab implementation of the methodology described in the supplied research text.

Pipeline:

1. Image enhancement: CLAHE + bilateral filtering
2. Adaptive fuzzy colour detection in HSV
3. SLIC superpixel generation
4. Superpixel feature extraction
5. Adaptive Fuzzy C-Means (AFCM)
6. Unsupervised convolutional autoencoder
7. Fuzzy attention-guided feature fusion
8. Traffic-sign shape identification
9. Symbol/number/text analysis
10. Multi-task traffic-sign analysis
11. Optional segmentation evaluation
12. Saving models and experimental results

Parameters not specified in the source methodology are exposed as configurable values.

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

!pip -q install scikit-image scikit-fuzzy opencv-python-headless matplotlib seaborn pandas tqdm pillow

print("Dependencies installed.")

In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import os
import cv2
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

from skimage import segmentation, morphology, feature
from skimage.segmentation import slic
from skimage.util import img_as_float

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CELL 3: GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 4: DATASET PATH
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/HUFAC_Net/dataset"

IMAGE_DIR = os.path.join(DATASET_ROOT, "images")
MASK_DIR = os.path.join(DATASET_ROOT, "masks")

print("Image directory:", IMAGE_DIR)
print("Exists:", os.path.exists(IMAGE_DIR))

if os.path.exists(MASK_DIR):
    print("Mask directory:", MASK_DIR)
else:
    print("No mask directory found. Unsupervised inference can still run.")

In [ ]:
# ============================================================
# CELL 5: FIND DATASET IMAGES
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

image_paths = []

for root, _, files in os.walk(IMAGE_DIR):
    for file in files:
        if Path(file).suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(os.path.join(root, file))

image_paths = sorted(image_paths)

print("Number of images:", len(image_paths))

if len(image_paths) == 0:
    raise RuntimeError("No images found. Check IMAGE_DIR.")

for p in image_paths[:10]:
    print(p)

In [ ]:
# ============================================================
# CELL 6: CONFIGURATION
# ============================================================

# Image enhancement
CLAHE_CLIP_LIMIT = 2.0
CLAHE_GRID = (8, 8)
BILATERAL_D = 9
BILATERAL_SIGMA_COLOR = 75
BILATERAL_SIGMA_SPACE = 75

# Fuzzy colour detection
FUZZY_THRESHOLD = 0.35

# SLIC
N_SEGMENTS = 400
COMPACTNESS = 12
SIGMA = 1

# AFCM
N_CLUSTERS = 3
FCM_M = 2.0
FCM_MAX_ITER = 150
FCM_EPSILON = 1e-5

# Autoencoder
AE_SIZE = 128
AE_BATCH_SIZE = 16
AE_EPOCHS = 30
AE_LR = 1e-3

RESULT_DIR = os.path.join(DATASET_ROOT, "HUFAC_results")
os.makedirs(RESULT_DIR, exist_ok=True)

print("Configuration loaded.")

In [ ]:
# ============================================================
# CELL 7: IMAGE LOADING
# ============================================================

def load_image(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

sample_image = load_image(image_paths[0])

print("Image shape:", sample_image.shape)

plt.figure(figsize=(10, 6))
plt.imshow(sample_image)
plt.axis("off")
plt.title("Original Traffic Image")
plt.show()

In [ ]:
# ============================================================
# CELL 8: IMAGE ENHANCEMENT
# ============================================================

def enhance_image(
    image,
    clip_limit=CLAHE_CLIP_LIMIT,
    tile_grid_size=CLAHE_GRID,
    bilateral_d=BILATERAL_D,
    sigma_color=BILATERAL_SIGMA_COLOR,
    sigma_space=BILATERAL_SIGMA_SPACE
):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )

    enhanced_l = clahe.apply(l_channel)
    enhanced_lab = cv2.merge([enhanced_l, a_channel, b_channel])

    enhanced_rgb = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2RGB
    )

    enhanced_rgb = cv2.bilateralFilter(
        enhanced_rgb,
        d=bilateral_d,
        sigmaColor=sigma_color,
        sigmaSpace=sigma_space
    )

    return enhanced_rgb

enhanced_image = enhance_image(sample_image)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(enhanced_image)
axes[1].set_title("CLAHE + Bilateral")
axes[1].axis("off")

plt.show()

In [ ]:
# ============================================================
# CELL 9: HSV + FUZZY MEMBERSHIP
# ============================================================

def rgb_to_hsv(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

def gaussian_membership(x, center, sigma):
    sigma = max(float(sigma), 1e-6)
    return np.exp(-((x - center) ** 2) / (2.0 * sigma ** 2))

def circular_hue_distance(h, center):
    d = np.abs(h - center)
    return np.minimum(d, 180.0 - d)

def circular_gaussian_membership(h, center, sigma):
    distance = circular_hue_distance(h, center)
    return np.exp(-(distance ** 2) / (2.0 * sigma ** 2))

COLOR_PARAMS = {
    "red":    dict(hue_center=0,   hue_sigma=10, sat_center=190, sat_sigma=75, val_center=150, val_sigma=100),
    "blue":   dict(hue_center=110, hue_sigma=15, sat_center=180, sat_sigma=75, val_center=140, val_sigma=100),
    "yellow": dict(hue_center=30,  hue_sigma=12, sat_center=180, sat_sigma=75, val_center=170, val_sigma=100),
    "green":  dict(hue_center=60,  hue_sigma=15, sat_center=160, sat_sigma=80, val_center=140, val_sigma=100),
    "white":  dict(hue_center=0,   hue_sigma=90, sat_center=30,  sat_sigma=45, val_center=210, val_sigma=60),
}

def compute_fuzzy_color_memberships(hsv):
    H = hsv[:, :, 0].astype(np.float32)
    S = hsv[:, :, 1].astype(np.float32)
    V = hsv[:, :, 2].astype(np.float32)

    memberships = {}

    for color_name, params in COLOR_PARAMS.items():
        if color_name == "white":
            hue_membership = np.ones_like(H)
        else:
            hue_membership = circular_gaussian_membership(
                H, params["hue_center"], params["hue_sigma"]
            )

        saturation_membership = gaussian_membership(
            S, params["sat_center"], params["sat_sigma"]
        )
        value_membership = gaussian_membership(
            V, params["val_center"], params["val_sigma"]
        )

        memberships[color_name] = (
            hue_membership *
            saturation_membership *
            value_membership
        )

    return memberships

def fuzzy_confidence_map(memberships):
    stacked = np.stack(list(memberships.values()), axis=0)
    confidence = np.max(stacked, axis=0)
    dominant_index = np.argmax(stacked, axis=0)
    color_names = list(memberships.keys())
    dominant_color = np.array(color_names, dtype=object)[dominant_index]
    return confidence, dominant_color

hsv_image = rgb_to_hsv(enhanced_image)
color_memberships = compute_fuzzy_color_memberships(hsv_image)
confidence_map, dominant_color = fuzzy_confidence_map(color_memberships)

plt.figure(figsize=(9, 6))
plt.imshow(confidence_map, cmap="jet", vmin=0, vmax=1)
plt.colorbar()
plt.title("Fuzzy Confidence Map")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 10: FUZZY CANDIDATE MASK
# ============================================================

candidate_mask = (confidence_map >= FUZZY_THRESHOLD).astype(np.uint8)

candidate_mask = morphology.remove_small_objects(
    candidate_mask.astype(bool),
    min_size=50
).astype(np.uint8)

kernel = np.ones((5, 5), np.uint8)
candidate_mask = cv2.morphologyEx(
    candidate_mask,
    cv2.MORPH_CLOSE,
    kernel
)

plt.figure(figsize=(10, 6))
plt.imshow(candidate_mask, cmap="gray")
plt.title("Fuzzy Candidate Traffic-Sign Mask")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 11: SLIC SUPERPIXELS
# ============================================================

image_float = img_as_float(enhanced_image)

segments = slic(
    image_float,
    n_segments=N_SEGMENTS,
    compactness=COMPACTNESS,
    sigma=SIGMA,
    start_label=0
)

print("Generated superpixels:", len(np.unique(segments)))

plt.figure(figsize=(12, 7))
plt.imshow(
    segmentation.mark_boundaries(
        enhanced_image, segments
    )
)
plt.title("SLIC Superpixels")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 12: SUPERPIXEL FEATURE EXTRACTION
# ============================================================

def calculate_entropy(region):
    region = np.asarray(region)
    if region.size == 0:
        return 0.0

    hist, _ = np.histogram(
        region,
        bins=32,
        range=(0, 255),
        density=True
    )
    hist = hist[hist > 0]

    if len(hist) == 0:
        return 0.0

    return -np.sum(hist * np.log2(hist + 1e-12))

def calculate_region_geometry(mask):
    area = np.sum(mask)
    if area < 1:
        return 0.0

    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0

    contour = max(contours, key=cv2.contourArea)
    perimeter = cv2.arcLength(contour, True)

    if perimeter <= 1e-6:
        return 0.0

    return float(4.0 * np.pi * area / (perimeter ** 2))

def extract_superpixel_features(rgb_image, hsv_image, segments):
    gray = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    gray_float = gray.astype(np.float32) / 255.0

    gx = cv2.Sobel(gray_float, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_float, cv2.CV_32F, 0, 1, ksize=3)
    gradient = np.sqrt(gx ** 2 + gy ** 2)

    lbp = feature.local_binary_pattern(
        gray, P=8, R=1, method="uniform"
    )

    feature_rows = []

    for segment_id in tqdm(
        np.unique(segments),
        desc="Extracting superpixel features"
    ):
        mask = segments == segment_id
        if not np.any(mask):
            continue

        h_values = hsv_image[:, :, 0][mask]
        s_values = hsv_image[:, :, 1][mask]
        v_values = hsv_image[:, :, 2][mask]
        gradient_values = gradient[mask]
        lbp_values = lbp[mask]
        gray_values = gray[mask]

        feature_rows.append({
            "segment_id": int(segment_id),
            "H": float(np.mean(h_values)),
            "S": float(np.mean(s_values)),
            "V": float(np.mean(v_values)),
            "gradient": float(np.mean(gradient_values)),
            "LBP": float(np.mean(lbp_values)),
            "entropy": float(calculate_entropy(gray_values)),
            "circularity": calculate_region_geometry(mask),
            "confidence": float(np.mean(confidence_map[mask]))
        })

    return pd.DataFrame(feature_rows)

feature_df = extract_superpixel_features(
    enhanced_image, hsv_image, segments
)

FEATURE_COLUMNS = [
    "H", "S", "V",
    "gradient", "LBP",
    "entropy", "circularity"
]

print(feature_df.head())
print("Feature shape:", feature_df.shape)

In [ ]:
# ============================================================
# CELL 13: ADAPTIVE FUZZY C-MEANS
# ============================================================

class AdaptiveFCM:
    def __init__(
        self,
        n_clusters=N_CLUSTERS,
        m=FCM_M,
        max_iter=FCM_MAX_ITER,
        epsilon=FCM_EPSILON,
        random_state=SEED
    ):
        self.n_clusters = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.epsilon = epsilon
        self.random_state = random_state
        self.centers_ = None
        self.membership_ = None
        self.objective_history_ = []

    def _initialise_membership(self, n_samples):
        rng = np.random.default_rng(self.random_state)
        U = rng.random((n_samples, self.n_clusters))
        U /= np.sum(U, axis=1, keepdims=True)
        return U

    def _update_centers(self, X, U):
        um = U ** self.m
        numerator = um.T @ X
        denominator = np.sum(um, axis=0)[:, None]
        return numerator / (denominator + 1e-12)

    def _update_membership(self, X, centers):
        distances = np.linalg.norm(
            X[:, None, :] - centers[None, :, :],
            axis=2
        )
        distances = np.maximum(distances, 1e-10)

        power = 2.0 / (self.m - 1.0)
        ratio = distances[:, :, None] / distances[:, None, :]
        denominator = np.sum(ratio ** power, axis=2)

        return 1.0 / denominator

    def _objective(self, X, U, centers):
        distances_squared = np.linalg.norm(
            X[:, None, :] - centers[None, :, :],
            axis=2
        ) ** 2

        return np.sum((U ** self.m) * distances_squared)

    def fit(self, X):
        U = self._initialise_membership(X.shape[0])
        previous_J = np.inf

        for iteration in range(self.max_iter):
            centers = self._update_centers(X, U)
            U = self._update_membership(X, centers)
            J = self._objective(X, U, centers)
            self.objective_history_.append(J)

            if abs(J - previous_J) < self.epsilon:
                print(f"FCM converged at iteration {iteration + 1}")
                break

            previous_J = J

        self.centers_ = centers
        self.membership_ = U
        return self

    def predict(self, X):
        U = self._update_membership(X, self.centers_)
        return np.argmax(U, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return np.argmax(self.membership_, axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(
    feature_df[FEATURE_COLUMNS].values.astype(np.float64)
)

fcm = AdaptiveFCM()
cluster_labels = fcm.fit_predict(X_scaled)
feature_df["cluster"] = cluster_labels

print(feature_df["cluster"].value_counts())

In [ ]:
# ============================================================
# CELL 14: AFCM CONVERGENCE
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(fcm.objective_history_, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Objective J")
plt.title("Adaptive Fuzzy C-Means Convergence")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# CELL 15: SUPERPIXEL LABELS -> SEGMENTATION MASK
# ============================================================

def superpixel_labels_to_mask(
    segments,
    labels,
    feature_df,
    confidence_threshold=0.20
):
    segmentation_mask = np.zeros(
        segments.shape,
        dtype=np.uint8
    )

    label_lookup = dict(
        zip(feature_df["segment_id"], labels)
    )

    confidence_lookup = dict(
        zip(
            feature_df["segment_id"],
            feature_df["confidence"]
        )
    )

    cluster_scores = {}
    for cluster_id in np.unique(labels):
        rows = feature_df[feature_df["cluster"] == cluster_id]
        cluster_scores[cluster_id] = rows["confidence"].mean()

    traffic_cluster = max(
        cluster_scores,
        key=cluster_scores.get
    )

    for segment_id in np.unique(segments):
        cluster = label_lookup.get(segment_id, -1)
        confidence = confidence_lookup.get(segment_id, 0)

        if (
            cluster == traffic_cluster and
            confidence >= confidence_threshold
        ):
            segmentation_mask[segments == segment_id] = 1

    return segmentation_mask, traffic_cluster

fcm_mask, traffic_cluster = superpixel_labels_to_mask(
    segments,
    cluster_labels,
    feature_df
)

fcm_mask = morphology.remove_small_objects(
    fcm_mask.astype(bool),
    min_size=100
).astype(np.uint8)

plt.figure(figsize=(10, 6))
plt.imshow(fcm_mask, cmap="gray")
plt.title("AFCM Initial Segmentation")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 16: REGION EXTRACTION
# ============================================================

def extract_candidate_regions(image, mask, min_area=150):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8),
        connectivity=8
    )

    regions = []

    for i in range(1, num_labels):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        if area < min_area:
            continue

        regions.append({
            "bbox": (x, y, w, h),
            "area": area,
            "image": image[y:y+h, x:x+w],
            "mask": (labels[y:y+h, x:x+w] == i).astype(np.uint8)
        })

    return regions

regions = extract_candidate_regions(
    enhanced_image, fcm_mask
)

print("Candidate regions:", len(regions))

visual = enhanced_image.copy()
for region in regions:
    x, y, w, h = region["bbox"]
    cv2.rectangle(
        visual, (x, y), (x+w, y+h),
        (255, 0, 0), 2
    )

plt.figure(figsize=(12, 8))
plt.imshow(visual)
plt.title("Candidate Traffic-Sign Regions")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 17: UNSUPERVISED CONVOLUTIONAL AUTOENCODER
# ============================================================

class RegionDataset(Dataset):
    def __init__(self, regions, image_size=AE_SIZE):
        self.regions = regions
        self.image_size = image_size

    def __len__(self):
        return len(self.regions)

    def __getitem__(self, idx):
        image = self.regions[idx]["image"]
        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        image = image.astype(np.float32) / 255.0

        return torch.from_numpy(
            image
        ).permute(2, 0, 1)

class HUFACAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        reconstructed = self.decode(z)
        return reconstructed, z

if len(regions) == 0:
    raise RuntimeError(
        "No candidate regions were detected. "
        "Adjust FUZZY_THRESHOLD or AFCM parameters."
    )

ae_dataset = RegionDataset(regions)
ae_loader = DataLoader(
    ae_dataset,
    batch_size=AE_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

autoencoder = HUFACAutoencoder().to(DEVICE)
optimizer = torch.optim.Adam(
    autoencoder.parameters(),
    lr=AE_LR
)
criterion = nn.MSELoss()

print(autoencoder)

In [ ]:
# ============================================================
# CELL 18: TRAIN AUTOENCODER
# ============================================================

loss_history = []
autoencoder.train()

for epoch in range(AE_EPOCHS):
    running_loss = 0.0

    for batch in tqdm(
        ae_loader,
        desc=f"Epoch {epoch+1}/{AE_EPOCHS}",
        leave=False
    ):
        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        reconstructed, _ = autoencoder(batch)

        loss = criterion(
            reconstructed,
            batch
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item() * batch.size(0)
        )

    epoch_loss = running_loss / len(ae_dataset)
    loss_history.append(epoch_loss)

    print(
        f"Epoch [{epoch+1}/{AE_EPOCHS}] "
        f"Loss: {epoch_loss:.6f}"
    )

plt.figure(figsize=(9, 5))
plt.plot(loss_history, linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Reconstruction Loss")
plt.title("HUFAC-Net Autoencoder Training")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# CELL 19: FUZZY ATTENTION + FEATURE FUSION
# ============================================================

def fuzzy_attention_map(memberships, weights=None):
    color_names = list(memberships.keys())
    membership_stack = np.stack(
        [memberships[name] for name in color_names],
        axis=0
    )

    K = membership_stack.shape[0]

    if weights is None:
        weights = np.ones(K)

    weights = np.asarray(weights, dtype=np.float32)

    numerator = np.sum(
        weights[:, None, None] * membership_stack,
        axis=0
    )

    denominator = np.sum(weights)

    return np.clip(
        numerator / (denominator + 1e-8),
        0, 1
    )

def fuse_attention_with_latent(latent, attention):
    batch_size = latent.size(0)
    height = latent.size(2)
    width = latent.size(3)

    attention_resized = cv2.resize(
        attention.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_LINEAR
    )

    attention_tensor = torch.from_numpy(
        attention_resized
    ).float().unsqueeze(0).unsqueeze(0).to(latent.device)

    attention_tensor = attention_tensor.repeat(
        batch_size, 1, 1, 1
    )

    fused = latent * attention_tensor

    return fused, attention_tensor

autoencoder.eval()

sample_batch = next(iter(ae_loader)).to(DEVICE)

with torch.no_grad():
    reconstructed, latent = autoencoder(sample_batch)

attention_map = fuzzy_attention_map(color_memberships)

fused_features, attention_tensor = fuse_attention_with_latent(
    latent,
    attention_map
)

print("Latent shape:", latent.shape)
print("Attention shape:", attention_tensor.shape)
print("Fused shape:", fused_features.shape)

fused_visual = (
    fused_features[0]
    .mean(dim=0)
    .detach()
    .cpu()
    .numpy()
)

plt.figure(figsize=(8, 6))
plt.imshow(fused_visual, cmap="jet")
plt.colorbar()
plt.title("Fuzzy Attention-Guided Fused Feature")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 20: SHAPE IDENTIFICATION
# ============================================================

def shape_descriptors(mask):
    mask = (mask.astype(np.uint8) * 255)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)

    if area <= 0 or perimeter <= 0:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)

    circularity = 4.0 * np.pi * area / (perimeter ** 2)
    compactness = perimeter ** 2 / (4.0 * np.pi * area)
    aspect_ratio = w / max(h, 1)

    solidity = area / max(hull_area, 1e-8)

    vertices = cv2.approxPolyDP(
        contour,
        0.04 * perimeter,
        True
    )

    return {
        "area": area,
        "perimeter": perimeter,
        "width": w,
        "height": h,
        "circularity": circularity,
        "compactness": compactness,
        "aspect_ratio": aspect_ratio,
        "solidity": solidity,
        "vertices": len(vertices)
    }

def classify_shape(desc):
    if desc is None:
        return "Unknown"

    v = desc["vertices"]
    c = desc["circularity"]
    ar = desc["aspect_ratio"]

    if c > 0.80:
        return "Circular"

    if v == 3:
        return "Triangular"

    if v == 8:
        return "Octagonal"

    if v == 4:
        return "Diamond" if 0.75 <= ar <= 1.33 else "Rectangular"

    if v == 5:
        return "Rectangular"

    return "Unknown"

shape_results = []

for i, region in enumerate(regions):
    desc = shape_descriptors(region["mask"])
    if desc is not None:
        shape_results.append({
            "region_id": i,
            "shape": classify_shape(desc),
            **desc
        })

shape_df = pd.DataFrame(shape_results)

display(shape_df)

In [ ]:
# ============================================================
# CELL 21: COLOUR + SYMBOL/NUMBER/TEXT ANALYSIS
# ============================================================

def region_dominant_color(region, memberships):
    x, y, w, h = region["bbox"]
    region_mask = region["mask"].astype(bool)

    scores = {}

    for name, membership in memberships.items():
        crop = membership[y:y+h, x:x+w]

        if crop.shape != region_mask.shape:
            scores[name] = 0
            continue

        values = crop[region_mask]
        scores[name] = float(np.mean(values)) if values.size else 0

    return max(scores, key=scores.get), scores

def segment_internal_content(sign_image, sign_mask):
    gray = cv2.cvtColor(sign_image, cv2.COLOR_RGB2GRAY)

    masked_gray = cv2.bitwise_and(
        gray,
        gray,
        mask=(sign_mask.astype(np.uint8) * 255)
    )

    _, binary = cv2.threshold(
        masked_gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    binary[sign_mask == 0] = 0

    binary = morphology.remove_small_objects(
        binary.astype(bool),
        min_size=5
    ).astype(np.uint8) * 255

    return binary

def component_features(binary):
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    components = []

    for i in range(1, n):
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        if area < 5:
            continue

        components.append({
            "label": i,
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "area": area,
            "aspect_ratio": w / max(h, 1),
            "fill_ratio": area / max(w * h, 1)
        })

    return components

def classify_component(component):
    w = component["width"]
    h = component["height"]
    ar = component["aspect_ratio"]
    fill = component["fill_ratio"]
    area = component["area"]

    if ar > 2.5 or ar < 0.40:
        return "Textual Information"

    if 0.25 <= ar <= 1.20 and 0.15 <= fill <= 0.75 and area > 15:
        return "Numerical Information"

    return "Graphical Symbol"

semantic_results = []

for region_id, region in enumerate(regions):
    binary = segment_internal_content(
        region["image"],
        region["mask"]
    )

    components = component_features(binary)

    for component in components:
        semantic_results.append({
            "region_id": region_id,
            "semantic_class": classify_component(component),
            **component
        })

semantic_df = pd.DataFrame(semantic_results)

display(semantic_df)

In [ ]:
# ============================================================
# CELL 22: INTEGRATED MULTI-TASK RESULT
# ============================================================

def analyse_region(region_id, region, memberships):
    desc = shape_descriptors(region["mask"])
    shape = classify_shape(desc)

    dominant_color, color_scores = region_dominant_color(
        region, memberships
    )

    binary = segment_internal_content(
        region["image"],
        region["mask"]
    )

    components = component_features(binary)

    semantic_classes = [
        classify_component(c)
        for c in components
    ]

    if semantic_classes:
        semantic_summary = (
            pd.Series(semantic_classes)
            .value_counts()
            .index[0]
        )
    else:
        semantic_summary = "None"

    return {
        "region_id": region_id,
        "bbox": region["bbox"],
        "dominant_colour": dominant_color,
        "shape": shape,
        "semantic_content": semantic_summary,
        "num_components": len(components)
    }

final_results = [
    analyse_region(i, region, color_memberships)
    for i, region in enumerate(regions)
]

final_df = pd.DataFrame(final_results)

display(final_df)

In [ ]:
# ============================================================
# CELL 23: FINAL VISUAL OUTPUT
# ============================================================

output = enhanced_image.copy()

for result in final_results:
    x, y, w, h = result["bbox"]

    label = (
        f'{result["dominant_colour"]} | '
        f'{result["shape"]}'
    )

    cv2.rectangle(
        output,
        (x, y),
        (x+w, y+h),
        (255, 0, 0),
        2
    )

    cv2.putText(
        output,
        label,
        (x, max(y-8, 15)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 0),
        2,
        cv2.LINE_AA
    )

plt.figure(figsize=(15, 9))
plt.imshow(output)
plt.title("HUFAC-Net Traffic-Sign Analysis")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# CELL 24: OPTIONAL SEGMENTATION METRICS
# ============================================================

def load_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {mask_path}")
    return (mask > 127).astype(np.uint8)

def calculate_segmentation_metrics(prediction, ground_truth):
    prediction = (prediction > 0).astype(np.uint8)
    ground_truth = (ground_truth > 0).astype(np.uint8)

    intersection = np.sum(prediction * ground_truth)

    union = (
        np.sum(prediction) +
        np.sum(ground_truth) -
        intersection
    )

    iou = intersection / (union + 1e-8)

    dice = (
        2.0 * intersection /
        (
            np.sum(prediction) +
            np.sum(ground_truth) +
            1e-8
        )
    )

    precision = intersection / (
        np.sum(prediction) + 1e-8
    )

    recall = intersection / (
        np.sum(ground_truth) + 1e-8
    )

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )

    return {
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

print("Metric function ready.")

In [ ]:
# ============================================================
# CELL 25: SAVE MODEL AND RESULTS
# ============================================================

output_path = os.path.join(
    RESULT_DIR,
    "hufac_output.png"
)

Image.fromarray(output).save(output_path)

model_path = os.path.join(
    RESULT_DIR,
    "HUFAC_autoencoder.pth"
)

torch.save(
    autoencoder.state_dict(),
    model_path
)

feature_df.to_csv(
    os.path.join(RESULT_DIR, "superpixel_features.csv"),
    index=False
)

shape_df.to_csv(
    os.path.join(RESULT_DIR, "shape_analysis.csv"),
    index=False
)

semantic_df.to_csv(
    os.path.join(RESULT_DIR, "semantic_components.csv"),
    index=False
)

final_df.to_csv(
    os.path.join(RESULT_DIR, "multi_task_results.csv"),
    index=False
)

print("Saved results to:", RESULT_DIR)
print("Output:", output_path)
print("Model:", model_path)

In [ ]:
# ============================================================
# CELL 26: END-TO-END INFERENCE FUNCTION
# ============================================================

def hufac_inference(image_path, display=True):
    image = load_image(image_path)

    enhanced = enhance_image(image)
    hsv = rgb_to_hsv(enhanced)

    memberships = compute_fuzzy_color_memberships(hsv)
    confidence, _ = fuzzy_confidence_map(memberships)

    segments_local = slic(
        img_as_float(enhanced),
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SIGMA,
        start_label=0
    )

    features_local = extract_superpixel_features(
        enhanced,
        hsv,
        segments_local
    )

    scaler_local = StandardScaler()

    X_local = scaler_local.fit_transform(
        features_local[FEATURE_COLUMNS].values.astype(np.float64)
    )

    fcm_local = AdaptiveFCM()

    labels_local = fcm_local.fit_predict(X_local)
    features_local["cluster"] = labels_local

    segmentation_mask, _ = superpixel_labels_to_mask(
        segments_local,
        labels_local,
        features_local
    )

    segmentation_mask = morphology.remove_small_objects(
        segmentation_mask.astype(bool),
        min_size=100
    ).astype(np.uint8)

    detected_regions = extract_candidate_regions(
        enhanced,
        segmentation_mask
    )

    results = [
        analyse_region(i, region, memberships)
        for i, region in enumerate(detected_regions)
    ]

    result_image = enhanced.copy()

    for result in results:
        x, y, w, h = result["bbox"]
        label = (
            f'{result["dominant_colour"]} | '
            f'{result["shape"]}'
        )

        cv2.rectangle(
            result_image,
            (x, y),
            (x+w, y+h),
            (255, 0, 0),
            2
        )

        cv2.putText(
            result_image,
            label,
            (x, max(y-5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

    if display:
        fig, axes = plt.subplots(
            2, 3,
            figsize=(18, 10)
        )

        axes[0, 0].imshow(image)
        axes[0, 0].set_title("Input")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(enhanced)
        axes[0, 1].set_title("Enhanced")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(confidence, cmap="jet")
        axes[0, 2].set_title("Fuzzy Confidence")
        axes[0, 2].axis("off")

        axes[1, 0].imshow(segmentation_mask, cmap="gray")
        axes[1, 0].set_title("AFCM Segmentation")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(
            segmentation.mark_boundaries(
                enhanced,
                segments_local
            )
        )
        axes[1, 1].set_title("SLIC Superpixels")
        axes[1, 1].axis("off")

        axes[1, 2].imshow(result_image)
        axes[1, 2].set_title("Final HUFAC-Net Output")
        axes[1, 2].axis("off")

        plt.tight_layout()
        plt.show()

    return {
        "enhanced_image": enhanced,
        "hsv": hsv,
        "memberships": memberships,
        "confidence_map": confidence,
        "segments": segments_local,
        "features": features_local,
        "fcm": fcm_local,
        "segmentation_mask": segmentation_mask,
        "regions": detected_regions,
        "results": pd.DataFrame(results),
        "final_image": result_image
    }

In [ ]:
# ============================================================
# CELL 27: TEST END-TO-END PIPELINE
# ============================================================

TEST_IMAGE = image_paths[0]

result = hufac_inference(
    TEST_IMAGE,
    display=True
)

display(result["results"])